# 🤝 Lab 4 — CPU–GPU Teamwork
**Foothold · GPU Computing for AI · Unit 4 · Book: Step 4**

How does the CPU hand work to the GPU? Today you do every step yourself: move data, allocate, copy, **write and launch your own kernels**, handle asynchronous launches, and finally profile and fix a slow training loop.

| Part | What you do | Goes with Foothold lesson |
|---|---|---|
| A | Where tensors live; measure PCIe, pinned vs normal memory | 4.1 Host & Device on PCIe |
| B | The five steps, one line at a time, timed | 4.2 The Five Steps of Offloading |
| C | Write ReLU and image kernels; block sizes; the `if (i < n)` check | 4.3 Kernels, Grids, Blocks, Threads |
| D | Asynchronous launches, honest timing, the cost of `.item()` | 4.4 Async Launch & Synchronization |
| E | Profile and fix a slow training loop | 4.C Capstone |

⏱️ About 2 hours (Parts A–C in one session, D–E in the next). 📘 Textbook: Kirk & Hwu Ch 2–3 · Hamacher §12.2.1, Ch 7 (PCIe), Ch 3 (interrupts). Course site: https://aishugithub.github.io/course_GPU/

💬 **Every line of code has a comment** (after `#` in Python, after `//` inside the CUDA C kernels).

## ▶️ How to run this notebook

1. **Save your own copy first:** menu **File → Save a copy in Drive**. Work in that copy — the original is read-only, and your copy is what you submit.
2. **Run cells in order, top to bottom.** Click a grey code cell and press **Shift + Enter** (or the ▶ button on its left). Wait for it to finish (a ✓ and the output appear) before running the next one.
3. **Switch on the GPU:** menu **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**. (This lab needs it. If Colab says no GPU is available, wait a while and try again — the free tier has limits.)
4. **Lost or confused?** Menu **Runtime → Restart session and run all** starts everything again from a clean slate.
5. Cells marked **✏️ YOUR TURN** are exercises: change the code, run it again, and write your answer in the text cell below it (double-click a text cell to edit it).

## 0 · Setup — run both cells

In [ ]:
import time, subprocess            # time: stopwatch; subprocess: run system commands
import numpy as np                  # arrays and maths on the CPU
import matplotlib.pyplot as plt     # charts
import torch                        # PyTorch: runs maths on the CPU or the GPU
from torch import nn                # nn: PyTorch's layers and losses

if torch.cuda.is_available():       # is there an NVIDIA GPU we can use? ("cuda" = NVIDIA's GPU system)
    device = "cuda"                 # yes: from now on, device = "cuda" means "put it on the GPU"
    print("✅ GPU found:", torch.cuda.get_device_name(0))   # the name of GPU number 0 (the first one)
else:                               # no GPU in this session
    device = "cpu"                  # fall back to the CPU
    print("⚠️ NO GPU! Go to Runtime → Change runtime type → T4 GPU → Save, then run this cell again.")   # how to get one
    print("   (Most of this notebook needs the GPU, so switch it on before going further.)")              # and why it matters

def sync():                         # GPU work runs in the background: wait for it to really finish before reading the clock
    if device == "cuda":            # (only needed when there is a GPU)
        torch.cuda.synchronize()    # wait until every queued GPU job is done

def timeit(fn, repeats=5):          # run fn several times and return the average time of one run, in seconds
    fn(); sync()                    # warm-up run (the first GPU call has extra start-up cost)
    t0 = time.perf_counter()        # start the stopwatch
    for _ in range(repeats):        # run it several times
        fn()                        # the work itself
    sync()                          # wait until the GPU has really finished all the runs
    return (time.perf_counter() - t0) / repeats   # total time ÷ number of runs

In [ ]:
# CuPy lets Python compile and run small CUDA C kernels. It comes pre-installed on Colab.
try:                                # "try": attempt this; if it fails, jump to "except" instead of crashing
    import cupy as cp               # CuPy: NumPy-like arrays on the GPU, plus our own CUDA kernels
    HAVE_CUPY = device == "cuda"    # we can only use it if there is also a GPU
    print("CuPy", cp.__version__, "ready" if HAVE_CUPY else "(installed, but no GPU in this session)")   # say whether it's usable
except ImportError:                 # CuPy is not installed
    HAVE_CUPY = False               # remember that, so later cells can say so
    print("CuPy not found. On Colab with a GPU, run:  !pip install cupy-cuda12x   then re-run this cell.")   # how to install it

def cuda_time_ms(launch, repeats=5):                # time GPU work with the GPU's own clock (CUDA events); returns ms per run
    launch(); cp.cuda.Device().synchronize()        # warm-up run, then wait for it to finish
    start, end = cp.cuda.Event(), cp.cuda.Event()   # two time-stamps on the GPU
    start.record()                                  # stamp "start" in the GPU's queue
    for _ in range(repeats):                        # several runs
        launch()                                    # queue the kernel once more
    end.record(); end.synchronize()                 # stamp "end", and wait until the GPU reaches it
    return cp.cuda.get_elapsed_time(start, end) / repeats   # time between the stamps ÷ runs

---
## Part A · Host and Device  *(Unit 4.1)*

Every tensor lives on one device. The GPU can only compute on tensors in its own memory.

In [ ]:
# 3 random numbers in CPU (host) memory, the default, and 3 made directly in GPU (device) memory
x_cpu = torch.rand(3)
x_gpu = torch.rand(3, device=device)   # 3 random numbers made directly in GPU memory
# .device tells you where a tensor lives
print("x_cpu lives on:", x_cpu.device)
print("x_gpu lives on:", x_gpu.device)   # and where this one lives

# try mixing the two; if it fails, show the first line of PyTorch's error instead of crashing
try:
    y = x_cpu + x_gpu   # mixing devices: this fails
except RuntimeError as e:   # e = the error PyTorch raised
    print("\nRuntimeError:", str(e).splitlines()[0])   # show the first line of the error message

# the fix: move the CPU tensor to the GPU first
y = x_cpu.to(device) + x_gpu
print("after moving:", y.device)   # now it works, on the GPU

**How fast is PCIe?** Copy tensors of growing size from host to device. Then repeat with **pinned** (page-locked) host memory, which the GPU's DMA engine can read directly. (A T4 is on PCIe Gen 3 ×16: about 16 GB/s on paper, about 12 GB/s in practice.)

In [ ]:
# copy sizes in megabytes, and a table heading
sizes_mb = [1, 10, 100, 500]
print(f"{'size':>8s} {'normal':>12s} {'pinned':>12s}")   # the table heading
for mb in sizes_mb:   # for each size
    # how many 4-byte floats make this many MB; one copy in ordinary memory, one in pinned memory
    n = mb * 1024 * 1024 // 4
    normal = torch.rand(n)   # a tensor in ordinary host memory
    pinned = normal.pin_memory()   # the same numbers in pinned (page-locked) host memory
    # time a host → device copy from each
    t_n = timeit(lambda: normal.to(device), repeats=3)
    t_p = timeit(lambda: pinned.to(device), repeats=3)   # ...and from pinned memory
    # speed = GB ÷ seconds (MB ÷ 1024 = GB)
    print(f"{mb:6d} MB {mb / 1024 / t_n:9.2f} GB/s {mb / 1024 / t_p:9.2f} GB/s")

### ✏️ YOUR TURN (A)
1. What PCIe speed did you measure? How does it compare with the GPU's own memory (measured in Lab 3)?
2. Why are small copies (1 MB) so much slower per byte than big ones?

*Your answers:*

---
## Part B · The Five Steps, One at a Time  *(Unit 4.2)*

ReLU on 100 MB of data, with every step as its own line and its own timer. `sync()` makes each timer honest.

In [ ]:
# 100 MB of float32 input data in host memory, and a dictionary of step name → time taken
n = 25_000_000
a_cpu = torch.randn(n)   # the input data, in host memory
steps = {}   # step name → time taken

# on each line below: do one step, sync() so the GPU really finishes, record the time, restart the clock
sync(); t = time.perf_counter()
# 1 · allocate space for the answer in GPU memory
b_gpu = torch.empty(n, device=device)
sync(); steps["1 allocate"] = time.perf_counter() - t; t = time.perf_counter()   # wait, record step 1, restart the clock
# 2 · copy the input in, across PCIe
a_gpu = a_cpu.to(device)
sync(); steps["2 copy in"] = time.perf_counter() - t; t = time.perf_counter()   # wait, record step 2, restart the clock
# 3 · launch the kernel: ReLU, written as "clamp below at 0"
torch.clamp(a_gpu, min=0, out=b_gpu)
sync(); steps["3 kernel"] = time.perf_counter() - t; t = time.perf_counter()   # wait, record step 3, restart the clock
# 4 · copy the answer out, across PCIe
b_cpu = b_gpu.cpu()
sync(); steps["4 copy out"] = time.perf_counter() - t; t = time.perf_counter()   # wait, record step 4, restart the clock
# 5 · free the GPU memory
del a_gpu, b_gpu
sync(); steps["5 free"] = time.perf_counter() - t   # wait, and record step 5

total = sum(steps.values())   # the time for all five steps together
for k, v in steps.items():   # for each step and its time
    # a text bar chart: up to 40 blocks per step, in proportion to its share of the time
    print(f"{k:11s} {v*1e3:8.2f} ms  {'█' * int(40 * v / total)}")
# compare with ReLU done on the CPU
print("Correct result:", torch.equal(b_cpu, torch.relu(a_cpu)))

The same five steps in **CuPy**. Notice how closely it mirrors the CUDA C version from the lesson (`cudaMalloc`, `cudaMemcpy`…).

In [ ]:
# the same five steps in CuPy, which mirrors CUDA C closely ("d_" = lives on the device)
a_host = np.random.randn(n).astype(np.float32)
# 1 allocate (cudaMalloc)
d_b = cp.empty(n, dtype=cp.float32)
# 2 copy in (cudaMemcpy host to device)
d_a = cp.asarray(a_host)
# 3 launch a kernel
cp.maximum(d_a, 0, out=d_b)
# 4 copy out (cudaMemcpy device to host)
b_host = cp.asnumpy(d_b)
# 5 free (cudaFree, through CuPy's memory pool)
del d_a, d_b
print("CuPy result correct:", np.allclose(b_host, np.maximum(a_host, 0)))   # check against ReLU on the CPU

### ✏️ YOUR TURN (B)
1. Which steps took the most time? What percentage was the actual computing (step 3)?
2. Change the code to run `torch.relu` **50 times** on `a_gpu` before copying out. What happens to the computing share?

*Your answers:*

---
## Part C · Write Your Own Kernels  *(Unit 4.3)*

Here's the ReLU kernel from the lesson, in CUDA C, compiled by CuPy. Each thread computes its global index `i` and handles one element.

In [ ]:
# the ReLU kernel in CUDA C (inside the C code, comments start with //)
relu_src = r'''
// a kernel: runs on the GPU, one copy per thread; x = input, y = output, n = how many elements
extern "C" __global__
void relu(const float *x, float *y, int n) {   // x = input array, y = output array, n = how many
    // this thread's global index
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    // the safety check, then ReLU: the larger of x[i] and 0
    if (i < n)
        y[i] = fmaxf(x[i], 0.0f);   // ReLU: the larger of x[i] and 0
}
'''
# CuPy compiles the C code into a GPU kernel
relu_k = cp.RawKernel(relu_src, "relu")
# deliberately NOT a multiple of any block size; input and output in GPU memory
n = 10_000_003
d_x = cp.random.randn(n, dtype=cp.float32); d_y = cp.empty_like(d_x)   # input and output arrays in GPU memory
block_ms = {}   # block size → time in ms
for threads in [32, 100, 256, 1024]:   # try four block sizes
    # round UP, so every element gets a thread
    blocks = (n + threads - 1) // threads
    # launch: kernel((blocks,), (threads per block,), (arguments)); cp.int32(n) passes n as a C int
    block_ms[threads] = cuda_time_ms(lambda: relu_k((blocks,), (threads,), (d_x, d_y, cp.int32(n))))
    # check against CuPy's own ReLU
    ok = bool(cp.allclose(d_y, cp.maximum(d_x, 0)))
    print(f"block size {threads:5d}: {blocks:7,d} blocks  {block_ms[threads]:6.3f} ms  correct={ok}")   # show the blocks, the time and the check

**What if you forget `if (i < n)`?** Writing past the end of a real array can crash the whole Colab session, so we do it safely: the output array gets 1,024 extra 'sentinel' slots set to −1, and we look at whether the kernel scribbled over them.

In [ ]:
unsafe_src = r'''
extern "C" __global__   // a kernel: runs on the GPU, launched from the CPU
void relu_unsafe(const float *x, float *y, int n) {   // the same ReLU, but without the safety check
    int i = blockIdx.x * blockDim.x + threadIdx.x;   // this thread's global index
    // NO check on y! (x[i % n] keeps the READ safe, so nothing crashes)
    y[i] = fmaxf(x[i % n], 0.0f);
}
'''
unsafe_k = cp.RawKernel(unsafe_src, "relu_unsafe")   # compile it
# 4 blocks of 256 = 1,024 threads for 1,000 elements
n, threads = 1000, 256
blocks = (n + threads - 1) // threads   # = 4 blocks
# all-positive input, and an output with 1,024 extra "sentinel" slots set to −1 after the real data
d_x = cp.random.rand(n, dtype=cp.float32) + 1
d_y = cp.full(n + 1024, -1.0, dtype=cp.float32)   # the output, plus 1,024 sentinel slots set to −1
# launch without the safety check, then count the sentinel slots that are no longer −1
unsafe_k((blocks,), (threads,), (d_x, d_y, cp.int32(n)))
damaged = int((d_y[n:] != -1).sum())   # count the sentinel slots no longer −1 (d_y[n:] = everything after the real data)
print(f"{blocks * threads} threads for {n} elements → {damaged} slots past the end were overwritten!")   # show the damage

**A 2-D kernel:** brighten an image, one thread per pixel, with 16 × 16 blocks.

In [ ]:
bright_src = r'''
extern "C" __global__   // a kernel: runs on the GPU, launched from the CPU
void brighten(float *img, int width, int height, float amount) {   // add "amount" to every pixel of a width × height image
    // this thread's column (x direction) and row (y direction)
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;   // and this thread's row
    // the safety check in both directions
    if (col < width && row < height)
        // pixel (row, col) is at position row × width + col; fminf caps it at 1.0 (white)
        img[row * width + col] = fminf(img[row * width + col] + amount, 1.0f);
}
'''
# a made-up 480 × 640 grey picture: a wavy pattern of brightness 0..0.8
H, W = 480, 640
yy, xx = np.mgrid[0:H, 0:W]   # the row and column number of every pixel
image = (0.4 * np.sin(xx / 40) * np.cos(yy / 30) + 0.4).astype(np.float32)   # a wavy pattern of grey levels

bright_k = cp.RawKernel(bright_src, "brighten")   # compile the kernel
# copy the image to the GPU; blocks of 16 × 16 = 256 threads; enough blocks across and down (rounded up)
d_img = cp.asarray(image)
block = (16, 16)   # blocks of 16 × 16 = 256 threads
grid = ((W + 15) // 16, (H + 15) // 16)   # blocks across and down: (40, 30)
# launch: add 0.35 to every pixel, then copy the result back
bright_k(grid, block, (d_img, cp.int32(W), cp.int32(H), cp.float32(0.35)))
result = cp.asnumpy(d_img)   # copy the result back to the CPU
print(f"grid = {grid} blocks of {block} threads = {grid[0] * grid[1] * 256:,} threads for {H * W:,} pixels")   # one thread per pixel

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))      # two pictures side by side
# vmin/vmax fix the grey scale to 0..1, so "before" and "after" use the same brightness scale
ax[0].imshow(image, cmap="gray", vmin=0, vmax=1); ax[0].set_title("before"); ax[0].axis("off")                        # the original
ax[1].imshow(result, cmap="gray", vmin=0, vmax=1); ax[1].set_title("after: brightened on the GPU"); ax[1].axis("off")  # the result
plt.show()                                          # draw them

### ✏️ YOUR TURN (C)
1. Which block size was slowest? Explain using warps and the 16-blocks-per-SM limit.
2. How many slots were overwritten without the check? Show the arithmetic: 4 × 256 − 1,000 = ?
3. Change the brighten kernel to make the image **darker**. Which single line did you change?

*Your answers:*

---
## Part D · Asynchronous Launches  *(Unit 4.4)*

First, see the "impossible" timing for yourself.

In [ ]:
# two big matrices on the GPU, and one warm-up multiply
a = torch.rand(4096, 4096, device=device); b = torch.rand(4096, 4096, device=device)
a @ b; sync()   # a warm-up multiply

# WRONG: stop the clock as soon as Python returns, when the GPU has only been GIVEN the order
t0 = time.perf_counter(); c = a @ b; t_wrong = time.perf_counter() - t0
# let that multiply finish before the next test
sync()
# RIGHT: wait for the GPU to finish (sync) before stopping the clock
t0 = time.perf_counter(); c = a @ b; sync(); t_right = time.perf_counter() - t0

# the work in one 4096 × 4096 multiply
flops = 2 * 4096**3
print(f"WITHOUT sync: {t_wrong*1e3:8.3f} ms  → 'would be' {flops / t_wrong / 1e12:8.1f} TFLOPS  (impossible)")   # the misleading number
print(f"WITH sync   : {t_right*1e3:8.3f} ms  → {flops / t_right / 1e12:8.1f} TFLOPS  (real)")   # the honest number

Now the cost of a **sync point every step**: train a small network for 300 steps, calling `loss.item()` every step vs only at the end.

In [ ]:
# a small network on the GPU, and one fixed batch of made-up data: 64 inputs of 1,024 numbers, 64 labels
net = torch.nn.Sequential(torch.nn.Linear(1024, 1024), torch.nn.ReLU(), torch.nn.Linear(1024, 10)).to(device)
opt = torch.optim.SGD(net.parameters(), lr=0.01)   # plain gradient descent
X = torch.randn(64, 1024, device=device); Y = torch.randint(0, 10, (64,), device=device)   # one fixed batch of made-up data, on the GPU
lossf = torch.nn.CrossEntropyLoss()   # the loss for "which class?" problems

def train(steps, item_every_step):   # train for some steps, with or without .item() every step
    # a running total that lives on the GPU, and a Python list for the every-step version
    total = torch.zeros((), device=device); log = []
    for _ in range(steps):   # one training step each time round
        # forward pass + loss, then slopes + update
        loss = lossf(net(X), Y)
        opt.zero_grad(); loss.backward(); opt.step()   # clear old slopes, backward pass, update
        if item_every_step:   # the version with a sync point every step
            # .item() copies the number to the CPU, so the CPU must wait for the GPU: a sync every step
            log.append(loss.item())
        else:   # the version without
            # .detach() keeps the value without its history; it stays on the GPU, no waiting
            total += loss.detach()
    # one sync at the end
    return total.item()

item_ms = {}   # True/False → time in ms
for flag in [True, False]:   # first with .item() every step, then without
    item_ms[flag] = timeit(lambda: train(300, flag), repeats=2) * 1e3   # time 300 steps (× 1e3 → ms)
    print(f"item() every step = {flag!s:5s}: {item_ms[flag]:7.1f} ms for 300 steps")   # show both

### ✏️ YOUR TURN (D)
1. How many times "faster than peak" was the unsynchronised timing? What was really being measured?
2. How much slower was calling `.item()` every step? (The gap grows when the CPU has other work to overlap, as in Part E.)

*Your answers:*

---
## Part E · Profile and Fix a Slow Training Loop  *(Unit 4.C)*

The final-year team's loop vs the fixed loop, on MNIST (the handwritten digits from Lab 1), one epoch each.

In [ ]:
from torch import nn   # nn: PyTorch's layers and losses
from torchvision import datasets, transforms   # ready-made datasets, and ways to convert images
from torch.utils.data import DataLoader   # DataLoader serves a dataset in batches
# scikit-learn's accuracy function (it works on CPU NumPy arrays)
from sklearn.metrics import accuracy_score

# MNIST handwritten digits, turned into tensors with pixel values 0..1
train_data = datasets.MNIST("data", train=True, download=True, transform=transforms.ToTensor())

def make_model():   # build a fresh copy of the network
    # the same starting weights for both loops: a fair race
    torch.manual_seed(0)
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 512), nn.ReLU(), nn.Linear(512, 512), nn.ReLU(), nn.Linear(512, 10)).to(device)   # 784 → 512 → 512 → 10, moved to the GPU

def slow_epoch(max_steps=None):   # one epoch of the team's loop (max_steps lets the profiler stop early)
    # the team's original loop: batches of 16, no worker processes
    model = make_model(); opt = torch.optim.SGD(model.parameters(), lr=0.05); lossf = nn.CrossEntropyLoss()
    loader = DataLoader(train_data, batch_size=16, shuffle=True)   # batches of 16, shuffled, no helper processes
    history = []   # the loss and accuracy of every step
    for step, (images, labels) in enumerate(loader):   # one batch at a time; enumerate also counts the steps
        images, labels = images.to(device), labels.to(device)   # copy the batch to the GPU
        scores = model(images); loss = lossf(scores, labels)   # forward pass and loss
        opt.zero_grad(); loss.backward(); opt.step()   # clear old slopes, backward pass, update
        # accuracy on the CPU: copies out and a sync, every step
        acc = accuracy_score(labels.cpu().numpy(), scores.argmax(1).cpu().numpy())
        # and another sync, every step
        history.append((loss.item(), acc))
        if max_steps and step + 1 >= max_steps: break   # stop early if asked
    return float(np.mean([h[0] for h in history])), float(np.mean([h[1] for h in history]))   # the epoch's average loss and accuracy

def fast_epoch(max_steps=None):   # one epoch of the fixed loop
    # the fixed loop: same model and data, better habits; a bigger batch can take a bigger learning rate
    model = make_model(); opt = torch.optim.SGD(model.parameters(), lr=0.3); lossf = nn.CrossEntropyLoss()
    # batches of 256; 2 worker processes prepare batches in parallel; pinned memory for faster copies
    loader = DataLoader(train_data, batch_size=256, shuffle=True, num_workers=2, pin_memory=True)
    # totals kept ON the GPU
    running = torch.zeros((), device=device); correct = torch.zeros((), device=device)
    for step, (images, labels) in enumerate(loader):   # one batch at a time; enumerate also counts the steps
        # non_blocking=True: start the copy and carry on without waiting for it
        images = images.to(device, non_blocking=True); labels = labels.to(device, non_blocking=True)
        scores = model(images); loss = lossf(scores, labels)   # forward pass and loss
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()   # clear old slopes (the cheapest way), backward pass, update
        # no .item(): nothing makes the CPU wait
        running += loss.detach(); correct += (scores.argmax(1) == labels).sum()
        if max_steps and step + 1 >= max_steps: break   # stop early if asked
    seen = min((step + 1) * 256, len(train_data))   # how many images we went through
    # ONE sync, at the very end
    return running.item() / (step + 1), correct.item() / seen

epoch_s = {}   # loop name → seconds per epoch
for name, fn in [("SLOW (team's loop)", slow_epoch), ("FAST (fixed loop)", fast_epoch)]:   # race the two loops
    # time one full epoch of each
    sync(); t0 = time.perf_counter(); out = fn(); sync()
    epoch_s[name] = time.perf_counter() - t0   # keep the time
    print(f"{name:20s}: {epoch_s[name]:6.1f} s per epoch   (epoch-average loss {out[0]:.3f}, accuracy {out[1]:.1%})")   # show the time, the loss and the accuracy

**Fair comparison note:** after one epoch the fast loop's accuracy may be a little lower. Batch 256 makes only **235 weight updates per epoch** compared with **3,750** for batch 16 (Unit 2.2: epochs, batches, iterations). Train the fast loop for 2–3 epochs and it catches up, still in less total time. Faster per epoch is only useful if the model still learns well, so always check both.

**The professional tool: the PyTorch profiler.** Profile 30 steps of each loop and compare the top entries. Look for `cudaMemcpyAsync` (copies) and `cudaStreamSynchronize` (the CPU waiting).

In [ ]:
# PyTorch's built-in profiler: record what the CPU and the GPU do
from torch.profiler import profile, ProfilerActivity
acts = [ProfilerActivity.CPU, ProfilerActivity.CUDA]   # record what the CPU and the GPU do
for name, fn in [("SLOW", slow_epoch), ("FAST", fast_epoch)]:   # profile both loops
    # everything inside this block is recorded: just 30 steps of the loop
    with profile(activities=acts) as prof:
        fn(max_steps=30)   # just 30 steps of the loop
    print(f"===== {name} loop, 30 steps: where the CPU's time went =====")   # a heading for this loop
    # the 6 operations that took the most CPU time (skipping the DataLoader's own row, which also
    # counts the time spent starting its helper processes)
    rows = [e for e in prof.key_averages() if not e.key.startswith("enumerate(DataLoader)")]
    for e in sorted(rows, key=lambda e: -e.self_cpu_time_total)[:6]:   # the 6 biggest, most time first
        print(f"  {e.key[:34]:34s} {e.self_cpu_time_total / 1e3:8.1f} ms   called {e.count:5d} times")   # its name, CPU time in ms, and how often it ran

### ✏️ YOUR TURN (E) — record your results
| | Slow loop | Fast loop |
|---|---|---|
| Seconds per epoch | | |
| Batch size | 16 | 256 |
| Sync points per step | | |

1. What speed-up did the fixes give?
2. In the profiler lists, name one entry that's big in the SLOW loop but small in the FAST loop, and explain it using a Unit 4 idea.
3. Run `!nvidia-smi` in a new cell **while** the slow loop runs (start it, then quickly run the other cell). What GPU-Util % do you see?

*Your answers:*

---
## 🎓 Wrap-up
- Host and device have separate memories joined by a narrow PCIe link: copy once, compute many times.
- Every GPU job: allocate → copy in → launch → copy out → free.
- Kernels: each thread finds its element with `blockIdx × blockDim + threadIdx`, guarded by `if (i < n)`.
- Launches are asynchronous: sync only when you need a value, and overlap CPU work with GPU work.
- **Next — Unit 5: Making AI Fast (Lab 5A).** Tiling, Tensor Cores, low precision, and the roofline.